# RWPE — Fixed Architecture, Node-Encoding Runs (HI-Small)

- **Question:** does a random-walk positional encoding (RWPE, 16 values per account telling the network where the account sits in the graph) help, when the GNN architecture is held *exactly* fixed?
- **Knobs per run:** `operator` (gin / gat / pna / transformer), `mp` / `readout` edge features as in the operator notebooks, `node_enc` = `rwpe16` (`rwpe8` later, by the rule in `EXPERIMENTS.md` §1). RWPE only widens `node_proj` to (6 + 16) → 128; nothing else changes.
- **Everything shared** (model template, training loop, threshold selection, metrics, plots, saving) lives in `gnn_core.py`; this notebook only sets the run list and paths. One operator per Kaggle session; each run is compared with the same configuration without RWPE in that operator's notebook.
- **Protocol:** threshold swept on validation each epoch, best-val-F1 checkpoint, test scored once; every metric on train / val / test. `invariant_params` must stay at the family value (GIN 67,587 · GATv2 67,585 · PNA 427,265 · Transformer 133,121).

## 0. Environment (Kaggle)

- Pinned PyTorch / PyG stack known to work on Kaggle GPU — do not change versions or order.
- Then clone (or pull) the public repo so the committed `gnn_core.py` is the one imported.

In [ ]:
!pip uninstall -y torch torchvision torchaudio torch-scatter torch-sparse pyg_lib
!pip install torch==2.8.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126
!pip install torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-2.8.0+cu126.html
!pip install torch_geometric

In [ ]:
import os
import sys

REPO_URL = 'https://github.com/sandrokhizanishvili/gnn_for_fraudulent_patterns.git'
REPO_DIR = '/kaggle/working/gnn_for_fraudulent_patterns'
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}
!git -C {REPO_DIR} log -1 --oneline
sys.path.append(REPO_DIR)

## 1. Configuration

- The only cell that differs from the operator notebooks: `CONFIGS` lists `(operator, mp, readout, node_enc)` per run; uncomment one operator's lines per Kaggle session.
- `RWPE_DIR`: the Kaggle dataset `hi-small-rwpe` (the six `Data/rwpe/rwpe_k{8,16}_{train,val,test}.pt` files), added as a second input next to `hi-small-gnn`.
- Runs land in `OUT_BASE/<OPERATOR>/<run>/`, mirroring `Outputs/<FAMILY>/` in the repo; the batch summary is `batch_summary_rwpe.csv`, so the family's own `batch_summary.csv` is untouched.

In [ ]:
# runs of this batch — one dict per run; one operator per Kaggle session
CONFIGS = [
    dict(operator='gin', mp='base', readout='full', node_enc='rwpe16'),           # GIN-5 + RWPE — compare with GIN-5
    dict(operator='gin', mp='full', readout='full', node_enc='rwpe16'),           # GIN-4 + RWPE — compare with GIN-4
    # dict(operator='gat', mp='base', readout='full', node_enc='rwpe16'),         # GAT-5 + RWPE — compare with GAT-5
    # dict(operator='gat', mp='full', readout='full', node_enc='rwpe16'),         # GAT-4 + RWPE — compare with GAT-4
    # dict(operator='pna', mp='base', readout='full', node_enc='rwpe16'),         # PNA-5 + RWPE — compare with PNA-5
    # dict(operator='pna', mp='full', readout='full', node_enc='rwpe16'),         # PNA-4 + RWPE — compare with PNA-4
    # dict(operator='transformer', mp='base', readout='full', node_enc='rwpe16'), # TR-5 + RWPE — compare with TR-5
    # dict(operator='transformer', mp='full', readout='full', node_enc='rwpe16'), # TR-4 + RWPE — compare with TR-4
]
MP_DIRECTION      = 'in'     # 'in' | 'bidirectional' — fixed for the whole batch
TEMPORAL_SAMPLING = False    # True samples only edges earlier than the seed (needs pyg-lib)

# paths
DATA_DIR    = '/kaggle/input/datasets/sandrokhizanishvili/hi-small-gnn'
RWPE_DIR    = '/kaggle/input/datasets/sandrokhizanishvili/hi-small-rwpe'   # Data/rwpe/ as a separate Kaggle dataset
OUT_BASE    = '/kaggle/working/Outputs'                                     # runs go to OUT_BASE/<OPERATOR>/, mirrors Outputs/<FAMILY>/

import gnn_core as core
for cfg in CONFIGS:
    os.makedirs(f"{OUT_BASE}/{cfg['operator'].upper()}", exist_ok=True)
print(f'torch {core.torch.__version__} | device {core.device} | gnn_core from {core.__file__}')

## 2. Load graphs

- Three cumulative PyG snapshots; context edges carry label −1, only `eval_mask` edges are scored.
- `col_sets` maps each knob value to `edge_attr` columns: base = first 20, gfp = next 61, full = all 81.

In [ ]:
graphs, col_sets, node_dim = core.load_graphs(DATA_DIR)

## 3. Model

- Fixed template (`gnn_core.EdgeClassifier`): `node_proj` → 2 message-passing layers (hidden 128, dropout 0.3, residual) → readout MLP on `[h_src ‖ h_dst ‖ e_seed]`; only the conv differs between operators.
- `core.set_node_encoding` appends the snapshot's RWPE columns to `graph.x`, so `node_proj` becomes (6 + 16) → 128 — the only tensor that changes against the run without RWPE.
- `invariant_params` counts `node_proj` at its base width 6 and must print the family value; PNA also needs the training-graph in-degree histogram.

In [ ]:
cfg = CONFIGS[0]
node_dim = core.set_node_encoding(graphs, cfg['node_enc'], RWPE_DIR)
deg_hist = core.in_degree_histogram(graphs['train']) if cfg['operator'] == 'pna' else None
model = core.build_model(cfg['operator'], node_dim, len(col_sets[cfg['mp']]), len(col_sets[cfg['readout']]),
                         MP_DIRECTION == 'bidirectional', deg_hist)
print(model)
print(f'node features {node_dim} | params {core.count_params(model):,} | invariant_params {core.invariant_params(model):,}   (config {cfg})')
del model

## 4. Run the batch

- One call per config: seed → model → loaders → 20 epochs → best checkpoint → all splits scored → `results.json`, `history.csv`, `curves.png`, `best.pt`, `predictions.csv` under `OUT_ROOT/<run>/`.
- `predictions.csv`: every evaluated edge of train / val / test with `edge_id` (= row in `Data/edge_features.csv`), `src`, `dst`, `edge_time`, `y`, `prob` (probability) and `pred` (class at the val-chosen threshold) — for the typology / error analyses.
- Per-epoch line: train / val loss, F1 (train at the val threshold), PR-AUC, seconds, RAM; `* new best` marks a checkpoint.

In [ ]:
rows = []
for cfg in CONFIGS:
    print('=' * 90)
    rows.append(core.run_experiment(cfg['operator'], cfg, graphs, col_sets, node_dim,
                                    f"{OUT_BASE}/{cfg['operator'].upper()}", MP_DIRECTION, TEMPORAL_SAMPLING,
                                    cfg['node_enc'], RWPE_DIR))
print('=' * 90)
print('batch complete')

## 5. Comparison across the batch

- `batch_summary_rwpe.csv` per operator holds every `<split>_<metric>` column plus `threshold`, `best_epoch`, `params`, `invariant_params`, `node_enc`.
- `invariant_params` must be identical in every row of an operator (asserted).

In [ ]:
for operator in sorted({cfg['operator'] for cfg in CONFIGS}):
    summary = core.summarize_batch([row for row in rows if row['operator'] == operator],
                                   f'{OUT_BASE}/{operator.upper()}', 'batch_summary_rwpe.csv')
    display(summary[['node_enc'] + core.DISPLAY_COLS].round(4))

## 6. Download

- Zip `OUT_BASE`; extract into `Outputs/` in the repo and commit the small result files (`best.pt` and `predictions.csv` stay out of git).

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/Outputs_RWPE', 'zip', OUT_BASE)